# 06 — HOD-Mamba-Hetero
**Reddit Opinion & Community Research — full multi-relation backbone + DMoN community detection**


In [1]:
import sys, importlib

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        get_ipython().system(f"pip install -q {pkg}")

for pkg, imp in [("torch", None), ("numpy", None), ("pandas", None), ("pyarrow", None), ("scikit-learn", "sklearn")]:
    ensure(pkg, imp)

print("Packages OK.")


Packages OK.


In [2]:
import os, json, pickle, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, classification_report

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = "/content/drive/MyDrive/PhD/reddit_opinion_research"
else:
    BASE_DIR = os.environ.get("REDDIT_RESEARCH_BASE_DIR", "./reddit_opinion_research")

with open(os.path.join(BASE_DIR, "processed_v2", "dataset", "dataset_contract.json"), "r", encoding="utf-8") as f:
    dataset_contract = json.load(f)
DATASET_DIR = dataset_contract["dataset_dir"]

with open(dataset_contract["vocab_path"], "r", encoding="utf-8") as f:
    vocab = json.load(f)

assert "relation_to_idx" in vocab, (
    "vocab.json has no relation_to_idx -- re-run the PATCHED notebook 04 before this notebook."
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Mounted at /content/drive
Using device: cuda


## Load dataset artifacts

In [3]:
def load_split(name):
    with open(os.path.join(DATASET_DIR, f"{name}_items.pkl"), "rb") as f:
        return pickle.load(f)

splits = {name: load_split(name) for name in ["train", "val", "test"]}
all_items = splits["train"] + splits["val"] + splits["test"]  # community detection isn't a train/test task

embeddings = np.load(dataset_contract["embeddings_path"]).astype("float32")
node_feat_matrix = np.load(os.path.join(DATASET_DIR, "node_feat_matrix.npy"))
static_graph = pd.read_parquet(dataset_contract["static_graph_path"])

sentiment_to_idx = vocab["sentiment_to_idx"]
emotion_to_idx = vocab["emotion_to_idx"]
idx_to_sentiment = {v: k for k, v in sentiment_to_idx.items()}
idx_to_emotion = {v: k for k, v in emotion_to_idx.items()}
N_SENTIMENT, N_EMOTION = len(sentiment_to_idx), len(emotion_to_idx)
N_RELATIONS = len(vocab["relation_to_idx"])
N_AUTHORS = node_feat_matrix.shape[0]

print(f"train/val/test: {len(splits['train'])}/{len(splits['val'])}/{len(splits['test'])} "
      f"({len(all_items)} total)")
print(f"embeddings: {embeddings.shape}, node_feat_matrix: {node_feat_matrix.shape}, N_RELATIONS: {N_RELATIONS}")
print(f"static_graph: {len(static_graph):,} edges, relations: {static_graph['relation'].value_counts().to_dict()}")


train/val/test: 6440/1380/1380 (9200 total)
embeddings: (210994, 768), node_feat_matrix: (22465, 6), N_RELATIONS: 3
static_graph: 1,807,555 edges, relations: {'co_participation': 1649602, 'reply_to_post': 79394, 'reply_to_comment': 78559}


## Collate + length-bucketed batching 

In [4]:
def make_collate_fn(embeddings, node_feat_matrix):
    emb_dim = embeddings.shape[1]
    n_node_feats = node_feat_matrix.shape[1]

    def collate(batch):
        B = len(batch)
        L = max(it["length"] for it in batch)
        emb_batch = np.zeros((B, L, emb_dim), dtype="float32")
        node_feat_batch = np.zeros((B, L, n_node_feats), dtype="float32")
        parent_batch = np.full((B, L), -1, dtype="int64")
        relation_batch = np.zeros((B, L), dtype="int64")
        author_batch = np.full((B, L), -1, dtype="int64")
        delta_t_batch = np.zeros((B, L), dtype="float32")
        sentiment_batch = np.full((B, L), -1, dtype="int64")
        emotion_batch = np.full((B, L), -1, dtype="int64")
        mask = np.zeros((B, L), dtype="bool")

        for i, it in enumerate(batch):
            n = it["length"]
            emb_batch[i, :n] = embeddings[it["emb_idx"]]
            valid_author = it["author_idx"] >= 0
            node_feat_batch[i, :n][valid_author] = node_feat_matrix[it["author_idx"][valid_author]]
            parent_batch[i, :n] = it["parent_idx"]
            relation_batch[i, :n] = it["relation_idx"]
            author_batch[i, :n] = it["author_idx"]
            delta_t_batch[i, :n] = it["delta_t"]
            sentiment_batch[i, :n] = it["sentiment_idx"]
            emotion_batch[i, :n] = it["emotion_idx"]
            mask[i, :n] = True

        return {
            "embeddings": torch.from_numpy(emb_batch), "node_features": torch.from_numpy(node_feat_batch),
            "parent_idx": torch.from_numpy(parent_batch), "relation_idx": torch.from_numpy(relation_batch),
            "author_idx": torch.from_numpy(author_batch), "delta_t": torch.from_numpy(delta_t_batch),
            "sentiment_idx": torch.from_numpy(sentiment_batch), "emotion_idx": torch.from_numpy(emotion_batch),
            "mask": torch.from_numpy(mask),
        }
    return collate

collate_fn = make_collate_fn(embeddings, node_feat_matrix)


def make_length_bucketed_batches(items, batch_size, rng):
    lengths = np.array([it["length"] for it in items])
    order = np.argsort(lengths)
    batches = [order[i:i + batch_size].tolist() for i in range(0, len(order), batch_size)]
    rng.shuffle(batches)
    return batches


## Relation-typed tree-recurrent SSM layer

In [5]:
class RelationTypedSSMLayer(nn.Module):
    def __init__(self, d_model, state_dim=16, n_relations=3):
        super().__init__()
        self.d_model, self.state_dim = d_model, state_dim
        self.relation_embed = nn.Embedding(n_relations, d_model)
        self.A_log = nn.Parameter(torch.randn(d_model, state_dim) * 0.1)
        self.W_delta = nn.Linear(d_model, d_model)
        self.W_B = nn.Linear(d_model, state_dim)
        self.W_C = nn.Linear(d_model, state_dim)
        self.D = nn.Parameter(torch.randn(d_model) * 0.1)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, parent_idx, mask, relation_idx):
        # x: (B, L, D); relation_idx: (B, L) int in [0, n_relations) -- 0 = root/none
        B, L, D = x.shape
        N = self.state_dim
        x_eff = x + self.relation_embed(relation_idx)  # condition the SSM's selection on relation type

        A = -torch.exp(self.A_log)
        delta = F.softplus(self.W_delta(x_eff))
        Bt = self.W_B(x_eff)
        Ct = self.W_C(x_eff)
        Abar = torch.exp(delta.unsqueeze(-1) * A.view(1, 1, D, N))
        Bbar = delta.unsqueeze(-1) * Bt.unsqueeze(2)

        batch_arange = torch.arange(B, device=x.device)
        h_history, y_list = [], []
        for t in range(L):
            valid_t = mask[:, t]
            p = parent_idx[:, t]
            has_parent = p >= 0
            if t == 0:
                h_prev = x.new_zeros(B, D, N)
            else:
                h_stack = torch.stack(h_history, dim=1)
                gathered = h_stack[batch_arange, p.clamp(min=0, max=t - 1)]
                h_prev = torch.where(has_parent.view(B, 1, 1), gathered, x.new_zeros(B, D, N))
            h_t = Abar[:, t] * h_prev + Bbar[:, t] * x_eff[:, t].unsqueeze(-1)
            h_t = torch.where(valid_t.view(B, 1, 1), h_t, torch.zeros_like(h_t))
            h_history.append(h_t)
            y_t = (Ct[:, t].unsqueeze(1) * h_t).sum(-1) + self.D * x_eff[:, t]
            y_t = torch.where(valid_t.view(B, 1), y_t, torch.zeros_like(y_t))
            y_list.append(y_t)

        y = torch.stack(y_list, dim=1)
        return self.norm(y + x)  # residual to the ORIGINAL x, not x_eff -- keeps the relation
                                   # conditioning as a per-layer modulation rather than an
                                   # accumulating shift across a deep stack


## HOD-Mamba-Hetero model

In [6]:
class HODMambaHetero(nn.Module):
    def __init__(self, emb_dim, node_feat_dim, d_model=128, state_dim=16, num_layers=4,
                 n_sentiment=3, n_emotion=5, n_relations=3):
        super().__init__()
        self.input_proj = nn.Linear(emb_dim, d_model)
        self.node_feat_proj = nn.Linear(node_feat_dim, d_model)
        self.layers = nn.ModuleList([
            RelationTypedSSMLayer(d_model, state_dim, n_relations) for _ in range(num_layers)
        ])
        self.root_context = nn.Parameter(torch.zeros(d_model))
        self.sentiment_head = nn.Linear(d_model, n_sentiment)
        self.emotion_head = nn.Linear(d_model, n_emotion)

    def encode(self, batch):
        x = self.input_proj(batch["embeddings"]) + self.node_feat_proj(batch["node_features"])
        parent_idx, mask, relation_idx = batch["parent_idx"], batch["mask"], batch["relation_idx"]
        for layer in self.layers:
            x = layer(x, parent_idx, mask, relation_idx)
        return x  # (B, L, D) final per-document representation, AFTER seeing each document

    def forward(self, batch):
        x = self.encode(batch)
        B, L, D = x.shape
        parent_idx = batch["parent_idx"]
        has_parent = parent_idx >= 0
        p = parent_idx.clamp(min=0)
        context = torch.stack([x[b, p[b]] for b in range(B)], dim=0)
        context = torch.where(has_parent.unsqueeze(-1), context, self.root_context.view(1, 1, D))
        return self.sentiment_head(context), self.emotion_head(context)


## Loss and evaluation helpers

In [7]:
def compute_loss(sent_logits, emo_logits, batch):
    parent_idx, mask = batch["parent_idx"], batch["mask"]
    forecastable = mask & (parent_idx >= 0)
    sent_valid = forecastable & (batch["sentiment_idx"] >= 0)
    emo_valid = forecastable & (batch["emotion_idx"] >= 0)
    loss = 0.0
    if sent_valid.any():
        loss = loss + F.cross_entropy(sent_logits[sent_valid], batch["sentiment_idx"][sent_valid])
    if emo_valid.any():
        loss = loss + F.cross_entropy(emo_logits[emo_valid], batch["emotion_idx"][emo_valid])
    return loss, sent_valid, emo_valid


@torch.no_grad()
def evaluate(model, items, batch_size=32):
    model.eval()
    rng = np.random.default_rng(0)
    batches = make_length_bucketed_batches(items, batch_size, rng)
    all_sp, all_st, all_ep, all_et = [], [], [], []
    for batch_idx in batches:
        batch = collate_fn([items[i] for i in batch_idx])
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        sl, el = model(batch)
        _, sv, ev = compute_loss(sl, el, batch)
        all_sp.append(sl[sv].argmax(-1).cpu().numpy()); all_st.append(batch["sentiment_idx"][sv].cpu().numpy())
        all_ep.append(el[ev].argmax(-1).cpu().numpy()); all_et.append(batch["emotion_idx"][ev].cpu().numpy())
    sp, st = np.concatenate(all_sp), np.concatenate(all_st)
    ep, et = np.concatenate(all_ep), np.concatenate(all_et)
    return (f1_score(st, sp, average="macro", zero_division=0),
            f1_score(et, ep, average="macro", zero_division=0), (st, sp), (et, ep))


## Phase 1: train the Hetero backbone 

In [8]:
model = HODMambaHetero(
    emb_dim=embeddings.shape[1], node_feat_dim=node_feat_matrix.shape[1],
    n_sentiment=N_SENTIMENT, n_emotion=N_EMOTION, n_relations=N_RELATIONS,
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

BATCH_SIZE, MAX_EPOCHS, PATIENCE = 32, 30, 5
rng = np.random.default_rng(42)
best_combined_f1, best_state, patience_ctr = -1.0, None, 0

for epoch in range(MAX_EPOCHS):
    t0 = time.time()
    model.train()
    batches = make_length_bucketed_batches(splits["train"], BATCH_SIZE, rng)
    total_loss = 0.0
    for batch_idx in batches:
        batch = collate_fn([splits["train"][i] for i in batch_idx])
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        sl, el = model(batch)
        loss, _, _ = compute_loss(sl, el, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    val_sent_f1, val_emo_f1, _, _ = evaluate(model, splits["val"], BATCH_SIZE)
    combined_f1 = (val_sent_f1 + val_emo_f1) / 2
    print(f"Epoch {epoch+1}/{MAX_EPOCHS} ({time.time()-t0:.1f}s): train_loss={total_loss/len(batches):.4f}, "
          f"val_sentiment_f1={val_sent_f1:.3f}, val_emotion_f1={val_emo_f1:.3f}, combined={combined_f1:.3f}")

    if combined_f1 > best_combined_f1:
        best_combined_f1, patience_ctr = combined_f1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}.")
            break

model.load_state_dict(best_state)
print(f"\nBest validation combined macro-F1: {best_combined_f1:.3f}")


Epoch 1/30 (38.5s): train_loss=2.3694, val_sentiment_f1=0.408, val_emotion_f1=0.235, combined=0.321
Epoch 2/30 (34.8s): train_loss=2.2569, val_sentiment_f1=0.422, val_emotion_f1=0.301, combined=0.361
Epoch 3/30 (35.2s): train_loss=2.2301, val_sentiment_f1=0.475, val_emotion_f1=0.305, combined=0.390
Epoch 4/30 (36.1s): train_loss=2.2030, val_sentiment_f1=0.442, val_emotion_f1=0.305, combined=0.373
Epoch 5/30 (34.9s): train_loss=2.1745, val_sentiment_f1=0.439, val_emotion_f1=0.305, combined=0.372
Epoch 6/30 (35.5s): train_loss=2.1573, val_sentiment_f1=0.448, val_emotion_f1=0.304, combined=0.376
Epoch 7/30 (35.0s): train_loss=2.1295, val_sentiment_f1=0.450, val_emotion_f1=0.311, combined=0.381
Epoch 8/30 (35.3s): train_loss=2.1048, val_sentiment_f1=0.447, val_emotion_f1=0.314, combined=0.381
Early stopping at epoch 8.

Best validation combined macro-F1: 0.390


In [9]:
test_sent_f1, test_emo_f1, (st, sp), (et, ep) = evaluate(model, splits["test"], BATCH_SIZE)
print(f"Test sentiment macro-F1: {test_sent_f1:.3f}  (Lite: 0.479)")
print(classification_report(st, sp, target_names=[idx_to_sentiment[i] for i in range(N_SENTIMENT)], zero_division=0))
print(f"\nTest emotion macro-F1: {test_emo_f1:.3f}  (Lite: 0.312)")
print(classification_report(et, ep, target_names=[idx_to_emotion[i] for i in range(N_EMOTION)], zero_division=0))


Test sentiment macro-F1: 0.482  (Lite: 0.479)
              precision    recall  f1-score   support

    Negative       0.58      0.65      0.61     11150
     Neutral       0.51      0.50      0.51     10236
    Positive       0.39      0.29      0.33      4922

    accuracy                           0.52     26308
   macro avg       0.49      0.48      0.48     26308
weighted avg       0.51      0.52      0.52     26308


Test emotion macro-F1: 0.307  (Lite: 0.312)
              precision    recall  f1-score   support

       Anger       0.48      0.55      0.51      6622
         Joy       0.33      0.47      0.39      5302
     Neutral       0.39      0.46      0.42      7866
     Sadness       0.36      0.15      0.21      4776
    Surprise       0.00      0.00      0.00      1742

    accuracy                           0.40     26308
   macro avg       0.31      0.33      0.31     26308
weighted avg       0.37      0.40      0.37     26308



## Phase 2: extract per-author embeddings from the trained (now frozen) backbone




In [10]:
@torch.no_grad()
def extract_user_embeddings(model, items, n_authors, d_model, batch_size=32):
    model.eval()
    sum_embed = torch.zeros(n_authors, d_model)
    count = torch.zeros(n_authors)
    rng = np.random.default_rng(0)
    batches = make_length_bucketed_batches(items, batch_size, rng)
    for batch_idx in batches:
        batch = collate_fn([items[i] for i in batch_idx])
        batch_gpu = {k: v.to(DEVICE) for k, v in batch.items()}
        reprs = model.encode(batch_gpu).cpu()
        mask, author_idx = batch["mask"], batch["author_idx"]
        valid = mask & (author_idx >= 0)
        sum_embed.index_add_(0, author_idx[valid], reprs[valid])
        count.index_add_(0, author_idx[valid], torch.ones(valid.sum().item()))
    count = count.clamp(min=1)
    return sum_embed / count.unsqueeze(1)

d_model = model.input_proj.out_features
user_embeddings = extract_user_embeddings(model, all_items, N_AUTHORS, d_model, BATCH_SIZE)
print("user_embeddings:", user_embeddings.shape)
print(f"Authors with at least one document (nonzero embedding): {(user_embeddings.abs().sum(dim=1) > 0).sum().item():,} / {N_AUTHORS:,}")


user_embeddings: torch.Size([22465, 128])
Authors with at least one document (nonzero embedding): 20,659 / 22,465


## Prepare the static author graph

In [11]:
normalized_weight = static_graph["weight"].astype("float64").copy()
for rel in static_graph["relation"].unique():
    rel_mask = static_graph["relation"] == rel
    total = normalized_weight[rel_mask].sum()
    if total > 0:
        normalized_weight[rel_mask] = normalized_weight[rel_mask] / total
static_graph["weight_norm"] = normalized_weight

src = torch.tensor(static_graph["source_idx"].values, dtype=torch.long)
dst = torch.tensor(static_graph["target_idx"].values, dtype=torch.long)
w = torch.tensor(static_graph["weight_norm"].values, dtype=torch.float32)

# symmetrize -- the stored graph is directed, community detection wants undirected
src_sym = torch.cat([src, dst]).to(DEVICE)
dst_sym = torch.cat([dst, src]).to(DEVICE)
w_sym = torch.cat([w, w]).to(DEVICE)
print(f"Symmetrized graph: {len(src_sym):,} directed edge entries")


Symmetrized graph: 3,615,110 directed edge entries


## DMoN community-detection head

In [12]:
class DMoNHead(nn.Module):
    def __init__(self, d_in, n_clusters):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_in, d_in), nn.ReLU(), nn.Linear(d_in, n_clusters))

    def forward(self, x):
        return torch.softmax(self.mlp(x), dim=-1)


def dmon_loss(C, src, dst, w):
    n, k = C.shape
    d = torch.zeros(n, device=C.device).index_add_(0, src, w)
    m2 = d.sum()
    edge_term = (w * (C[src] * C[dst]).sum(dim=1)).sum()
    Cd = (C * d.unsqueeze(1)).sum(dim=0)
    degree_term = (Cd @ Cd) / m2
    L_mod = -(1.0 / m2) * (edge_term - degree_term)
    collapse_reg = (torch.sqrt(torch.tensor(float(k), device=C.device)) / n) * torch.norm(C.sum(dim=0), p=2) - 1
    return L_mod, collapse_reg


@torch.no_grad()
def hard_modularity(cluster_ids, src, dst, w):
    n = cluster_ids.shape[0]
    d = torch.zeros(n, device=cluster_ids.device).index_add_(0, src, w)
    m2 = d.sum()
    same_cluster = (cluster_ids[src] == cluster_ids[dst]).float()
    edge_term = (w * same_cluster).sum()
    k = int(cluster_ids.max().item()) + 1
    cluster_degree_sum = torch.zeros(k, device=cluster_ids.device).index_add_(0, cluster_ids, d)
    degree_term = (cluster_degree_sum ** 2).sum() / m2
    return ((edge_term - degree_term) / m2).item()


## Train DMoN 

In [13]:
N_CLUSTERS = 12  # exploratory default -- worth comparing modularity across a few values of K later

dmon_input = torch.cat([user_embeddings, torch.from_numpy(node_feat_matrix)], dim=1).to(DEVICE)
dmon_head = DMoNHead(dmon_input.shape[1], N_CLUSTERS).to(DEVICE)
dmon_optimizer = torch.optim.Adam(dmon_head.parameters(), lr=1e-2, weight_decay=1e-4)

DMON_EPOCHS = 300
for epoch in range(DMON_EPOCHS):
    dmon_optimizer.zero_grad()
    C = dmon_head(dmon_input)
    L_mod, L_collapse = dmon_loss(C, src_sym, dst_sym, w_sym)
    loss = L_mod + L_collapse
    loss.backward()
    dmon_optimizer.step()
    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            hard_q = hard_modularity(C.argmax(dim=1), src_sym, dst_sym, w_sym)
        print(f"DMoN epoch {epoch+1}/{DMON_EPOCHS}: L_mod={L_mod.item():.4f}, "
              f"L_collapse={L_collapse.item():.4f}, hard_modularity(Q)={hard_q:.4f}")

cluster_ids = dmon_head(dmon_input).argmax(dim=1)
final_q = hard_modularity(cluster_ids, src_sym, dst_sym, w_sym)
cluster_sizes = torch.bincount(cluster_ids, minlength=N_CLUSTERS)
print(f"\nFinal modularity Q = {final_q:.4f}")
print("Cluster sizes:", cluster_sizes.tolist())


DMoN epoch 50/300: L_mod=-0.0000, L_collapse=0.0000, hard_modularity(Q)=0.0247
DMoN epoch 100/300: L_mod=-0.0042, L_collapse=0.0001, hard_modularity(Q)=0.0706
DMoN epoch 150/300: L_mod=-0.0848, L_collapse=0.0040, hard_modularity(Q)=0.0950
DMoN epoch 200/300: L_mod=-0.0954, L_collapse=0.0029, hard_modularity(Q)=0.1080
DMoN epoch 250/300: L_mod=-0.1018, L_collapse=0.0033, hard_modularity(Q)=0.1126
DMoN epoch 300/300: L_mod=-0.1049, L_collapse=0.0025, hard_modularity(Q)=0.1149

Final modularity Q = 0.1152
Cluster sizes: [2386, 2478, 1301, 2988, 1486, 1933, 651, 1266, 1574, 1945, 2147, 2310]


## The modularity score




In [14]:
MODELS_DIR = os.path.join(BASE_DIR, "processed_v2", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(MODELS_DIR, "hod_mamba_hetero.pt"))
torch.save(dmon_head.state_dict(), os.path.join(MODELS_DIR, "dmon_head.pt"))
np.save(os.path.join(MODELS_DIR, "user_cluster_ids.npy"), cluster_ids.cpu().numpy())

results = {
    "model": "HOD-Mamba-Hetero",
    "test_sentiment_macro_f1": float(test_sent_f1),
    "test_emotion_macro_f1": float(test_emo_f1),
    "lite_comparison": {"sentiment_macro_f1": 0.479, "emotion_macro_f1": 0.312},
    "n_clusters": N_CLUSTERS,
    "final_modularity_q": float(final_q),
    "cluster_sizes": cluster_sizes.tolist(),
}
with open(os.path.join(MODELS_DIR, "hod_mamba_hetero_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(json.dumps(results, indent=2, ensure_ascii=False))


{
  "model": "HOD-Mamba-Hetero",
  "test_sentiment_macro_f1": 0.482493665455113,
  "test_emotion_macro_f1": 0.30698786119594423,
  "lite_comparison": {
    "sentiment_macro_f1": 0.479,
    "emotion_macro_f1": 0.312
  },
  "n_clusters": 12,
  "final_modularity_q": 0.11515481770038605,
  "cluster_sizes": [
    2386,
    2478,
    1301,
    2988,
    1486,
    1933,
    651,
    1266,
    1574,
    1945,
    2147,
    2310
  ]
}


# **Louvain baseline**

In [15]:
import networkx as nx

G = nx.Graph()
G.add_nodes_from(range(N_AUTHORS))
edge_weight_dict = {}
for s, d, wt in zip(src_sym.cpu().tolist(), dst_sym.cpu().tolist(), w_sym.cpu().tolist()):
    if s == d:
        continue
    key = (min(s, d), max(s, d))
    edge_weight_dict[key] = edge_weight_dict.get(key, 0.0) + wt
for (s, d), wt in edge_weight_dict.items():
    G.add_edge(s, d, weight=wt)

louvain_communities = nx.community.louvain_communities(G, weight="weight", seed=42)
louvain_cluster_ids = torch.zeros(N_AUTHORS, dtype=torch.long, device=src_sym.device)
for cid, comm in enumerate(louvain_communities):
    for node in comm:
        louvain_cluster_ids[node] = cid

louvain_q = hard_modularity(louvain_cluster_ids, src_sym, dst_sym, w_sym)
print(f"Louvain: {len(louvain_communities)} communities found automatically, Q = {louvain_q:.4f}")
print(f"DMoN:    {N_CLUSTERS} communities (fixed K), Q = {final_q:.4f}")

results["louvain_n_communities"] = len(louvain_communities)
results["louvain_modularity_q"] = float(louvain_q)
np.save(os.path.join(MODELS_DIR, "louvain_cluster_ids.npy"), louvain_cluster_ids.cpu().numpy())
with open(os.path.join(MODELS_DIR, "hod_mamba_hetero_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

Louvain: 1033 communities found automatically, Q = 0.2675
DMoN:    12 communities (fixed K), Q = 0.1152


In [16]:
print("Louvain dendrogram (natural granularity levels, each optimizing standard modularity):")
dendrogram_levels = list(nx.community.louvain_partitions(G, weight="weight", seed=42))
level_summaries = []
for level in dendrogram_levels:
    level_ids = torch.zeros(N_AUTHORS, dtype=torch.long, device=src_sym.device)
    for cid, comm in enumerate(level):
        for node in comm:
            level_ids[node] = cid
    q = hard_modularity(level_ids, src_sym, dst_sym, w_sym)
    level_summaries.append({"n_communities": len(level), "modularity_q": float(q)})
    print(f"  {len(level):6d} communities -> Q={q:.4f}")

closest_level = min(level_summaries, key=lambda s: abs(s["n_communities"] - N_CLUSTERS))
print(f"\nClosest to DMoN's K={N_CLUSTERS}: {closest_level['n_communities']} communities, Q={closest_level['modularity_q']:.4f}")
print(f"DMoN:                       {N_CLUSTERS} communities, Q={final_q:.4f}")

results["louvain_dendrogram"] = level_summaries
results["louvain_closest_to_dmon_k"] = closest_level
with open(os.path.join(MODELS_DIR, "hod_mamba_hetero_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

Louvain dendrogram (natural granularity levels, each optimizing standard modularity):
    2627 communities -> Q=0.1913
    1050 communities -> Q=0.2644
    1033 communities -> Q=0.2675

Closest to DMoN's K=12: 1033 communities, Q=0.2675
DMoN:                       12 communities, Q=0.1152


# **Fixed-K baseline**

If the closest dendrogram level above is nowhere near 12, that's telling you something real -- this graph's modularity-optimal granularity is much finer than 12, and there's no way to force Louvain's own hierarchy into an interpretable-scale partition without leaving the standard-modularity objective it's actually optimizing (the resolution-parameter route was tried and rejected earlier for exactly this reason).

The direct fix: stop trying to make Louvain match K=12, and instead ask the sharper question -- does DMoN's graph-aware objective beat naive clustering on the exact same input features, at the exact same K? K-means on dmon_input (the same backbone embeddings + node features DMoN sees) with no awareness of the graph at all isolates whether the modularity-based training objective specifically is earning its keep, independent of any granularity-matching problem.

In [17]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit(dmon_input.cpu().numpy())
kmeans_ids = torch.tensor(kmeans.labels_, dtype=torch.long, device=src_sym.device)
kmeans_q = hard_modularity(kmeans_ids, src_sym, dst_sym, w_sym)

print(f"K-means on the same features (K={N_CLUSTERS}, graph-blind):        Q = {kmeans_q:.4f}")
print(f"DMoN (K={N_CLUSTERS}, graph-aware modularity objective):           Q = {final_q:.4f}")
print(f"Louvain (unconstrained, {len(louvain_communities)} communities):{' '*(20-len(str(len(louvain_communities))))}Q = {louvain_q:.4f}")

results["kmeans_same_k_modularity_q"] = float(kmeans_q)
with open(os.path.join(MODELS_DIR, "hod_mamba_hetero_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

K-means on the same features (K=12, graph-blind):        Q = 0.0225
DMoN (K=12, graph-aware modularity objective):           Q = 0.1152
Louvain (unconstrained, 1033 communities):                Q = 0.2675
